# Turbo Evaluation with Masks (Colab)

This notebook evaluates anomaly detection from ZIP bundles already in Google Drive.

## Inputs (Drive)
- `val_fast.zip` (IXI validation, healthy; masks optional)
- `eval_test_with_masks.zip` (BraTS evaluation with images/masks/labels)

## What this notebook does
1. Extracts ZIPs into Colab runtime (`/content/local_data/...`) like a turbo loader.
2. Runs inference with ECNN checkpoint.
3. Computes **classification metrics** (AUROC, AUPRC, Accuracy, Precision, Recall, Specificity, F1, FPR, FNR).
4. Computes **reconstruction metrics** (MSE, MAE, SSIM; overall + normal/lesion splits).
5. Computes **lesion localization metrics** (Dice/IoU on lesion-positive slices).
6. Visualizes best anomalies using reconstruction + Gaussian-smoothed error + masks.

## Outputs (Drive)
- `evaluations/tables/turbo_eval_metrics_summary.csv`
- `evaluations/tables/turbo_eval_score_comparisons.csv`
- `evaluations/tables/turbo_eval_per_slice_metrics.csv`

In [ ]:
import os
import re
import sys
import json
import zipfile
import subprocess
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.ndimage import gaussian_filter
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# Colab drive mount
from google.colab import drive
drive.mount('/content/drive')

# Package checks
def ensure_pkg(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

ensure_pkg('skimage', 'scikit-image')
ensure_pkg('e2cnn', 'e2cnn')

from skimage.metrics import structural_similarity as ssim

In [ ]:
# Repo + imports for ECNN loader
REPO_URL = 'https://github.com/RifaDeen/symAD-ECNN.git'
REPO_ROOT = Path('/content/symAD-ECNN')

if not REPO_ROOT.exists():
    print('Cloning repo...')
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_ROOT)])
else:
    print('Repo exists; pulling latest...')
    subprocess.check_call(['git', '-C', str(REPO_ROOT), 'pull'])

EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
ECNN_DIR = EVALS_DIR / 'ecnn_thresholding'
for p in [REPO_ROOT, EVALS_DIR, ECNN_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from ecnn_model_loader import get_model_for_inference

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Paths: zips in Drive + runtime extraction targets
DRIVE_ROOT = Path('/content/drive/MyDrive/symAD-ECNN')
DATA_ROOT = DRIVE_ROOT / 'data'
OUT_DIR = DRIVE_ROOT / 'evaluations' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer your current zip names
IXI_ZIP_CANDIDATES = [
    DATA_ROOT / 'eval_ixi_fast.zip',
    DATA_ROOT / 'val_fast.zip',
]
BRATS_ZIP_CANDIDATES = [
    DATA_ROOT / 'brats_test_eval_fast.zip',
    DATA_ROOT / 'eval_test_with_masks.zip',
]

IXI_ZIP = next((p for p in IXI_ZIP_CANDIDATES if p.exists()), None)
BRATS_ZIP = next((p for p in BRATS_ZIP_CANDIDATES if p.exists()), None)

RUNTIME_ROOT = Path('/content/local_data')
IXI_EXTRACT = RUNTIME_ROOT / 'ixi_val'
BRATS_EXTRACT = RUNTIME_ROOT / 'brats_eval'

print(f'IXI zip:   {IXI_ZIP}')
print(f'BraTS zip: {BRATS_ZIP}')
assert IXI_ZIP is not None, f'Missing IXI zip. Checked: {IXI_ZIP_CANDIDATES}'
assert BRATS_ZIP is not None, f'Missing BraTS zip. Checked: {BRATS_ZIP_CANDIDATES}'

In [ ]:
def extract_zip(zip_path: Path, dst: Path, clean=True):
    if clean and dst.exists():
        import shutil
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {zip_path.name} -> {dst}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dst)

extract_zip(IXI_ZIP, IXI_EXTRACT, clean=True)
extract_zip(BRATS_ZIP, BRATS_EXTRACT, clean=True)
print('Extraction complete.')

In [ ]:
def resolve_data_root(base: Path):
    """
    Returns a dict with detected paths for images/masks/labels.
    Supports both:
    1) nested layout: root/images, root/masks, root/labels
    2) flat layout: root/*.npy (images only)
    """
    candidates = [base, base / 'processed'] + [p for p in base.rglob('*') if p.is_dir()]

    # nested first
    for c in candidates:
        images = c / 'images'
        if images.exists() and len(list(images.glob('*.npy'))) > 0:
            return {
                'root': c,
                'images_dir': images,
                'masks_dir': c / 'masks',
                'labels_dir': c / 'labels',
                'layout': 'nested'
            }

    # flat fallback
    for c in candidates:
        imgs = [p for p in c.glob('*.npy') if not p.name.startswith('label_')]
        if len(imgs) > 0:
            return {
                'root': c,
                'images_dir': c,
                'masks_dir': c / 'masks',
                'labels_dir': c / 'labels',
                'layout': 'flat'
            }

    return None

IXI_INFO = resolve_data_root(IXI_EXTRACT)
BRATS_INFO = resolve_data_root(BRATS_EXTRACT)

print(f'IXI info:   {IXI_INFO}')
print(f'BraTS info: {BRATS_INFO}')
assert IXI_INFO is not None, 'Could not find IXI data structure.'
assert BRATS_INFO is not None, 'Could not find BraTS data structure.'

def count_npy(d):
    return len(list(d.glob('*.npy'))) if d.exists() else 0

print('--- Counts ---')
for name, info in [('IXI', IXI_INFO), ('BraTS', BRATS_INFO)]:
    print(name, f"(layout={info['layout']})")
    print('  images:', count_npy(info['images_dir']))
    print('  masks :', count_npy(info['masks_dir']))
    print('  labels:', count_npy(info['labels_dir']))

In [ ]:
def sid_from_name(stem: str):
    m = re.search(r'(\d+)$', stem)
    return m.group(1) if m else stem

def _is_valid_npy(fp: Path):
    try:
        if fp is None or (not fp.exists()) or fp.stat().st_size == 0:
            return False
        arr = np.load(fp)
        return isinstance(arr, np.ndarray) and arr.size > 0
    except Exception:
        return False

def build_records(info: dict, source: str, force_label=None):
    images_dir = info['images_dir']
    masks_dir = info['masks_dir']
    labels_dir = info['labels_dir']
    layout = info['layout']

    image_files = sorted(images_dir.glob('*.npy'))
    records = []
    skipped_bad_image = 0

    for ip in image_files:
        name = ip.name
        stem = ip.stem
        sid = sid_from_name(stem)

        # skip label files in flat layouts
        if name.startswith('label_'):
            continue

        # skip corrupted/empty image files
        if not _is_valid_npy(ip):
            skipped_bad_image += 1
            continue

        # mask path candidates
        mp = None
        if masks_dir.exists():
            for cand in [masks_dir / name, masks_dir / f'slice_{sid}.npy']:
                if cand.exists() and _is_valid_npy(cand):
                    mp = cand
                    break

        # label assignment
        label = None
        if force_label is not None:
            label = int(force_label)
        else:
            # Try labels folder
            if labels_dir.exists():
                for cand in [labels_dir / f'label_{sid}.npy', labels_dir / name.replace('slice_', 'label_')]:
                    if cand.exists() and _is_valid_npy(cand):
                        try:
                            label = int(np.load(cand)[0])
                        except Exception:
                            label = None
                        break

            # If still missing, infer from mask if available
            if label is None and mp is not None:
                try:
                    m = np.load(mp)
                    label = int((m > 0).sum() > 0)
                except Exception:
                    label = None

            # final fallback
            if label is None:
                label = 0

        records.append({
            'id': sid,
            'image_path': ip,
            'mask_path': mp,
            'label': int(label),
            'source': source,
            'layout': layout,
        })

    print(f"{source}: skipped bad/empty images = {skipped_bad_image}")
    return records

def validate_records(records):
    """Final defensive pass to avoid EOFError during DataLoader iteration."""
    good = []
    bad = 0
    for r in tqdm(records, desc='Validating records', leave=False):
        ip = Path(r['image_path'])
        if not _is_valid_npy(ip):
            bad += 1
            continue

        mp = r.get('mask_path', None)
        if mp is not None and (not _is_valid_npy(Path(mp))):
            r = dict(r)
            r['mask_path'] = None

        good.append(r)

    print(f"Validation complete: kept {len(good)} / {len(records)} | dropped {bad}")
    return good

# IXI has no labels in your zip -> force healthy=0
ixi_records = build_records(IXI_INFO, 'ixi_val', force_label=0)
brats_records = build_records(BRATS_INFO, 'brats_eval', force_label=None)
all_records = validate_records(ixi_records + brats_records)

print(f'IXI records:   {len(ixi_records)} (forced healthy=0)')
print(f'BraTS records: {len(brats_records)}')
print(f'Combined:      {len(all_records)}')
print('Label distribution:', dict(pd.Series([r['label'] for r in all_records]).value_counts().sort_index()))

In [ ]:
class CombinedEvalDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = np.load(r['image_path']).astype(np.float32)
        x = torch.from_numpy(img).unsqueeze(0).float()

        if r['mask_path'] is not None and Path(r['mask_path']).exists():
            mask = np.load(r['mask_path']).astype(np.uint8)
        else:
            mask = np.zeros_like(img, dtype=np.uint8)

        return (
            x,
            x.clone(),
            int(r['label']),
            r['id'],
            torch.from_numpy(mask).unsqueeze(0).float()
        )

def normalize01(x):
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-8:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def dice_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    return float((2.0 * inter) / (pred.sum() + gt.sum() + 1e-8))

def iou_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    union = ((pred + gt) > 0).sum()
    return float(inter / (union + 1e-8))

def threshold_from_normals(scores, labels, target_fpr=0.20):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    ns = scores[labels == 0]
    if len(ns) == 0:
        return float(np.percentile(scores, 80))
    return float(np.percentile(ns, (1 - target_fpr) * 100))

In [ ]:
# Load ECNN model checkpoint
CKPT_CANDIDATES = [
    DRIVE_ROOT / 'models' / 'saved_models' / 'ecnn_optimized' / 'ecnn_optimized_best.pth',
    REPO_ROOT / 'models' / 'saved_models' / 'ecnn_optimized' / 'ecnn_optimized_best.pth',
]

ckpt_path = next((p for p in CKPT_CANDIDATES if p.exists()), None)
if ckpt_path is None:
    raise FileNotFoundError('ECNN checkpoint not found in expected locations.')

model, _ = get_model_for_inference(str(ckpt_path), str(device))
model = model.to(device).eval()
print(f'Loaded checkpoint: {ckpt_path}')

In [ ]:
@torch.no_grad()
def run_inference(model, records, batch_size=32, pixel_percentile=95, smooth_sigma=1.0):
    ds = CombinedEvalDataset(records)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    # store
    ids = []
    labels = []
    mask_pixels = []
    source_col = []

    mse_all, mae_all, ssim_all = [], [], []
    score_raw_mse_mean = []
    score_raw_mae_mean = []
    score_smooth_mse_mean = []
    score_smooth_mae_mean = []
    score_inv_ssim = []

    lesion_dice, lesion_iou = [], []

    for x, target, lab, sid, mask in tqdm(dl, desc='Inference'):
        x = x.to(device)
        target = target.to(device)
        mask = mask.to(device)

        recon = model(x)
        if recon.shape[-2:] != target.shape[-2:]:
            recon = F.interpolate(recon, size=target.shape[-2:], mode='bilinear', align_corners=False)

        err = recon - target
        err_abs = torch.abs(err)
        err_sq = err ** 2

        # batch-level scalar scores
        raw_mse_mean = err_sq.mean(dim=[1,2,3]).detach().cpu().numpy()
        raw_mae_mean = err_abs.mean(dim=[1,2,3]).detach().cpu().numpy()

        # smoothing for anomaly maps/scores
        err_abs_np = err_abs.detach().cpu().numpy()[:, 0]
        err_sq_np = err_sq.detach().cpu().numpy()[:, 0]

        smooth_abs = np.stack([gaussian_filter(e, sigma=smooth_sigma) for e in err_abs_np], axis=0)
        smooth_sq = np.stack([gaussian_filter(e, sigma=smooth_sigma) for e in err_sq_np], axis=0)

        smooth_mse_mean = smooth_sq.reshape(smooth_sq.shape[0], -1).mean(axis=1)
        smooth_mae_mean = smooth_abs.reshape(smooth_abs.shape[0], -1).mean(axis=1)

        # SSIM
        t_np = target.detach().cpu().numpy()
        r_np = recon.detach().cpu().numpy()
        batch_ssim = []
        for b in range(t_np.shape[0]):
            gt = t_np[b, 0].astype(np.float32)
            rc = r_np[b, 0].astype(np.float32)
            dr = float(max(gt.max(), rc.max()) - min(gt.min(), rc.min()))
            dr = dr if dr > 1e-8 else 1.0
            batch_ssim.append(float(ssim(gt, rc, data_range=dr)))

        inv_ssim = 1.0 - np.array(batch_ssim, dtype=np.float32)

        # lesion metrics using smoothed abs error map
        m_np = (mask.detach().cpu().numpy() > 0).astype(np.uint8)
        for b in range(smooth_abs.shape[0]):
            gt_mask = m_np[b, 0]
            if gt_mask.sum() > 0:
                emap = normalize01(smooth_abs[b])
                thr = np.percentile(emap, pixel_percentile)
                pred = (emap >= thr).astype(np.uint8)
                lesion_dice.append(dice_score(pred, gt_mask))
                lesion_iou.append(iou_score(pred, gt_mask))

        # reconstruction metrics
        mse_all.extend(raw_mse_mean.tolist())
        mae_all.extend(raw_mae_mean.tolist())
        ssim_all.extend(batch_ssim)

        # score candidates
        score_raw_mse_mean.extend(raw_mse_mean.tolist())
        score_raw_mae_mean.extend(raw_mae_mean.tolist())
        score_smooth_mse_mean.extend(smooth_mse_mean.tolist())
        score_smooth_mae_mean.extend(smooth_mae_mean.tolist())
        score_inv_ssim.extend(inv_ssim.tolist())

        # ids/meta
        ids.extend(list(sid))
        labels.extend(lab.numpy().tolist())
        source_col.extend(['unknown'] * len(lab))
        mask_pixels.extend([int(v) for v in m_np[:,0].reshape(m_np.shape[0], -1).sum(axis=1)])

    per_slice = pd.DataFrame({
        'slice_id': ids,
        'label': np.asarray(labels, dtype=int),
        'mask_pixels': np.asarray(mask_pixels, dtype=int),
        'mse': np.asarray(mse_all, dtype=float),
        'mae': np.asarray(mae_all, dtype=float),
        'ssim': np.asarray(ssim_all, dtype=float),
        'score_raw_mse_mean': np.asarray(score_raw_mse_mean, dtype=float),
        'score_raw_mae_mean': np.asarray(score_raw_mae_mean, dtype=float),
        'score_smooth_mse_mean': np.asarray(score_smooth_mse_mean, dtype=float),
        'score_smooth_mae_mean': np.asarray(score_smooth_mae_mean, dtype=float),
        'score_1_minus_ssim': np.asarray(score_inv_ssim, dtype=float),
    })

    return per_slice, lesion_dice, lesion_iou

per_slice_df, lesion_dice, lesion_iou = run_inference(model, all_records, batch_size=32, pixel_percentile=95, smooth_sigma=1.0)
print(per_slice_df.head())
print('Total slices:', len(per_slice_df))

In [ ]:
labels = per_slice_df['label'].to_numpy().astype(int)

score_cols = [
    'score_raw_mse_mean',
    'score_raw_mae_mean',
    'score_smooth_mse_mean',
    'score_smooth_mae_mean',
    'score_1_minus_ssim',
]

score_rows = []
for c in score_cols:
    s = per_slice_df[c].to_numpy().astype(float)
    if np.unique(labels).size > 1:
        auroc = float(roc_auc_score(labels, s))
        auprc = float(average_precision_score(labels, s))
    else:
        auroc, auprc = np.nan, np.nan

    normal_mask = labels == 0
    lesion_mask = labels == 1
    gap = float(np.mean(s[lesion_mask]) - np.mean(s[normal_mask])) if normal_mask.any() and lesion_mask.any() else np.nan

    score_rows.append({'score_name': c, 'auroc': auroc, 'auprc': auprc, 'gap_lesion_minus_normal': gap})

score_cmp_df = pd.DataFrame(score_rows).sort_values('auroc', ascending=False).reset_index(drop=True)
display(score_cmp_df)

best_score_name = score_cmp_df.loc[0, 'score_name']
best_scores = per_slice_df[best_score_name].to_numpy().astype(float)

thr = threshold_from_normals(best_scores, labels, target_fpr=0.20)
pred = (best_scores >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0,1]).ravel()

summary = {
    'model': 'ECNN-AE (Optimized)',
    'selected_score': best_score_name,
    'threshold': float(thr),
    'n_samples': int(len(labels)),
    'n_normal': int((labels==0).sum()),
    'n_lesion': int((labels==1).sum()),

    'auroc': float(roc_auc_score(labels, best_scores)) if np.unique(labels).size > 1 else np.nan,
    'auprc': float(average_precision_score(labels, best_scores)) if np.unique(labels).size > 1 else np.nan,
    'accuracy': float(accuracy_score(labels, pred)),
    'precision': float(precision_score(labels, pred, zero_division=0)),
    'recall': float(recall_score(labels, pred, zero_division=0)),
    'specificity': float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan,
    'f1_score': float(f1_score(labels, pred, zero_division=0)),
    'fpr': float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
    'fnr': float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan,
    'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),

    'mean_mse_all': float(per_slice_df['mse'].mean()),
    'mean_mae_all': float(per_slice_df['mae'].mean()),
    'mean_ssim_all': float(per_slice_df['ssim'].mean()),
    'mean_mse_normal': float(per_slice_df.loc[per_slice_df['label']==0, 'mse'].mean()) if (per_slice_df['label']==0).any() else np.nan,
    'mean_mse_lesion': float(per_slice_df.loc[per_slice_df['label']==1, 'mse'].mean()) if (per_slice_df['label']==1).any() else np.nan,
    'mean_mae_normal': float(per_slice_df.loc[per_slice_df['label']==0, 'mae'].mean()) if (per_slice_df['label']==0).any() else np.nan,
    'mean_mae_lesion': float(per_slice_df.loc[per_slice_df['label']==1, 'mae'].mean()) if (per_slice_df['label']==1).any() else np.nan,
    'mean_ssim_normal': float(per_slice_df.loc[per_slice_df['label']==0, 'ssim'].mean()) if (per_slice_df['label']==0).any() else np.nan,
    'mean_ssim_lesion': float(per_slice_df.loc[per_slice_df['label']==1, 'ssim'].mean()) if (per_slice_df['label']==1).any() else np.nan,

    'mean_dice_lesion_only': float(np.mean(lesion_dice)) if len(lesion_dice) else np.nan,
    'mean_iou_lesion_only': float(np.mean(lesion_iou)) if len(lesion_iou) else np.nan,
    'n_lesion_for_dice_iou': int(len(lesion_dice)),
}

summary_df = pd.DataFrame([summary])
display(summary_df.T)

In [ ]:
# Visualize best anomaly cases (top 5) with requested panel
# Panel: Input | Reconstruction (+error value) | Error Map | Smooth Error Map | Seg Mask

record_by_id = {str(r['id']): r for r in all_records}

def concentration_index(x2d, top_frac=0.01):
    v = x2d.reshape(-1)
    k = max(1, int(len(v) * top_frac))
    top_mean = np.partition(v, len(v)-k)[len(v)-k:].mean()
    base = v.mean() + 1e-8
    return float(top_mean / base)

# Select lesion slices only, then prioritize by highest selected anomaly score
cand = per_slice_df[per_slice_df['label'] == 1].copy()

# Add concentration on smooth error map for clearer, focused anomalies
conc_vals = []
for _, row in cand.iterrows():
    sid = str(row['slice_id'])
    rec = record_by_id[sid]
    img = np.load(rec['image_path']).astype(np.float32)
    x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad():
        recon = model(x).detach().cpu().numpy()[0, 0]
    err_map = np.abs(img - recon).astype(np.float32)
    smooth_err = gaussian_filter(err_map, sigma=1.0).astype(np.float32)
    conc_vals.append(concentration_index(smooth_err, top_frac=0.01))

cand['concentration'] = np.array(conc_vals, dtype=float)

# Rank: highest score first, then highest concentration
best_df = cand.sort_values([best_score_name, 'concentration'], ascending=[False, False]).head(5)

def infer_one(rec, smooth_sigma=1.0):
    img = np.load(rec['image_path']).astype(np.float32)
    mask = np.load(rec['mask_path']).astype(np.uint8) if rec['mask_path'] is not None and Path(rec['mask_path']).exists() else np.zeros_like(img, dtype=np.uint8)

    x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad():
        recon = model(x).detach().cpu().numpy()[0, 0]

    err_map = np.abs(img - recon).astype(np.float32)
    smooth_err = gaussian_filter(err_map, sigma=smooth_sigma).astype(np.float32)

    return img, recon, err_map, smooth_err, mask

if len(best_df) == 0:
    print('No lesion-positive slices found. Cannot visualize best anomaly cases.')
else:
    n = len(best_df)
    fig, axes = plt.subplots(n, 5, figsize=(18, 3.5 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)

    for r, (_, row) in enumerate(best_df.iterrows()):
        sid = str(row['slice_id'])
        rec = record_by_id[sid]

        img, recon, err_map, smooth_err, gt_mask = infer_one(rec, smooth_sigma=1.0)

        recon_err = float(np.mean((img - recon) ** 2))
        selected_score_val = float(row[best_score_name])
        conc = float(row['concentration'])

        axes[r, 0].imshow(img, cmap='gray')
        axes[r, 0].set_title(f'Input\n{sid}')
        axes[r, 0].axis('off')

        axes[r, 1].imshow(recon, cmap='gray')
        axes[r, 1].set_title(f'Reconstruction\nMSE={recon_err:.6f} | score={selected_score_val:.6f}')
        axes[r, 1].axis('off')

        axes[r, 2].imshow(err_map, cmap='hot')
        axes[r, 2].set_title('Error Map')
        axes[r, 2].axis('off')

        axes[r, 3].imshow(smooth_err, cmap='hot')
        axes[r, 3].set_title(f'Smooth Error\nconc={conc:.2f}')
        axes[r, 3].axis('off')

        axes[r, 4].imshow(gt_mask, cmap='gray')
        axes[r, 4].set_title(f'Seg Mask\nmask_pixels={int(gt_mask.sum())}')
        axes[r, 4].axis('off')

    plt.tight_layout()
    plt.show()

    print('Displayed top 5 BEST anomaly lesion slices (highest score + most concentrated maps).')

In [ ]:
summary_path = OUT_DIR / 'turbo_eval_metrics_summary.csv'
score_cmp_path = OUT_DIR / 'turbo_eval_score_comparisons.csv'
per_slice_path = OUT_DIR / 'turbo_eval_per_slice_metrics.csv'

summary_df.to_csv(summary_path, index=False)
score_cmp_df.to_csv(score_cmp_path, index=False)
per_slice_df.to_csv(per_slice_path, index=False)

print('Saved:')
print(' -', summary_path)
print(' -', score_cmp_path)
print(' -', per_slice_path)

print('\nDone.')